In [11]:
import numpy as np
import matplotlib.pyplot as plt
from GWFish.modules.fft import fft_lal_timeseries
from GWFish import detection
from scipy.interpolate import interp1d
from scipy.integrate import simpson
import math
import lal
from lal import REAL8TimeSeries, CreateREAL8TimeSeries, REAL8Vector, CreateREAL8Vector

In [12]:
filename = "/mnt/c/Users/ludov/OneDrive/Desktop/SN/GWFish/test_data/23_gwstrain_trim.dat"

In [13]:
#change to dataframe
params = {
    "ra" : np.random.uniform(0, 2*np.pi),
    "dec" : np.random.uniform(0, 2*np.pi),
    "psi" : np.random.uniform(0, 2*np.pi),
    "max_frequency_cutoff" : 2048
}

geo_time = 1395964818 #GPS time
distance = 10 #kpc
kpc = 3.086e21 #cm
dims = 300_000

In [14]:
def make_fft_from_time_series(time_series_input, df, dt, title="Ines_Ludo"):
    '''
    Returns the FFT done through the lal library given a time series. Also returns the frequency range.
    time_series_input: array of the time series 
    df: frequency step
    dt: time step
    title: title of the time series (optional)
    '''
    dims = len(time_series_input)
    time_series = CreateREAL8Vector(dims)
    time_series.data = time_series_input
    ts = CreateREAL8TimeSeries(title, 1, 0, dt, lal.DimensionlessUnit, dims)
    ts.data = time_series
    fft_dat = fft_lal_timeseries(ts, df).data.data
    freq_range = np.linspace( 0, df * len(fft_dat), len(fft_dat) )
    return fft_dat, freq_range

def frequency_plot_options(ax, fig, y_bounds = [1e-25, 1e-22], x_bounds = [1, 1e4]):
    ax.set_yscale("log")
    ax.set_ylabel(f"$|\\tilde{{h}}_{{+/x}}|$")
    ax.set_xlabel("f[Hz]")
    ax.set_xscale("log")
    ax.set_ylim(y_bounds)
    ax.set_xlim(x_bounds)
    ax.grid()
    ax.legend()
    fig.tight_layout()
    return 0


GW_dat = np.loadtxt(filename).T

new_time = np.linspace(min(GW_dat[0]), max(GW_dat[0]), dims)
interpolated_data = interp1d(GW_dat[0], GW_dat[1:], axis=1)(new_time)
GW_dat_interp = interpolated_data / (distance * kpc)

fft_h_plus = np.fft.fft(GW_dat_interp[0], dims, norm="forward" ) 
fft_h_cross = np.fft.fft(GW_dat_interp[1], dims, norm="forward" )
frequencies = np.fft.fftfreq(dims, d=np.mean(np.diff(new_time)))

df = frequencies[1] - frequencies[0]
dt = new_time[1] - new_time[0]

fft_dat_plus, freq_range = make_fft_from_time_series(GW_dat_interp[0], df, dt)
fft_dat_cross, _ = make_fft_from_time_series(GW_dat_interp[1], df, dt)

time_dim = dims//2+1
timevector = np.ones( time_dim ) * geo_time
detector = detection.Detector("ET")

phi_in = np.exp(1.j*(2*detector.frequencyvector*np.pi*geo_time)).T[0] #TODO shape is (dims, 1) makes it too high dimensional
fft_dat_plus = phi_in*np.conjugate( fft_dat_plus )
fft_dat_cross = phi_in*np.conjugate( fft_dat_cross )

# GW Fish format for hfp and hfc
hfp = fft_dat_plus[:, np.newaxis]
hfc = fft_dat_cross[:, np.newaxis]
polarizations = np.hstack((hfp, hfc))

args = (params, detector, polarizations, timevector)

signal = detection.projection(*args)
frequencyvector = detector.frequencyvector

frequencyvector = freq_range

component_SNRs = detection.SNR(detector, signal, frequencyvector=np.squeeze(frequencyvector))
out_SNR = np.sqrt(np.sum(component_SNRs**2))
print(out_SNR)

# timevector = np.ones * geocent_time 1980 GPS in seconds
# Gaussian ra dec
# Psi 0, 2pi flat 
# 0, 2pi flat phase

<Swig Object of type 'tagCOMPLEX16FrequencySeries *' at 0x7f3e69d80230>
<Swig Object of type 'tagCOMPLEX16FrequencySeries *' at 0x7f3e69d63af0>
Entered Detector.__init__
Loading detector configuration from /mnt/c/Users/ludov/OneDrive/Desktop/SN/GWFish/GWFish/detectors.yaml
npoints:  150001
115.72696501097523
